Installing Libraries

In [ ]:
!pip install -q -U google-genai

import pandas as pd
import json, time, random, re, os
from google import genai
from google.genai import types

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.1/259.1 kB 12.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.56.3 which is incompatible.


Imports

In [ ]:
import pandas as pd
import json, time, random, re, os
from google import genai
from google.genai import types

Model Config

In [ ]:
from google.colab import userdata
API_KEY = userdata.get('Teachermodel')

client = genai.Client(api_key=API_KEY)
MODEL_ID = "gemini-3.5-flash"

GEN_CONFIG = types.GenerateContentConfig(
    temperature=1.0,
    max_output_tokens=800,
    thinking_config=types.ThinkingConfig(thinking_budget=0)
)

CRITIC_CONFIG = types.GenerateContentConfig(
    temperature=0.0,          # deterministic for verification
    max_output_tokens=20,
    thinking_config=types.ThinkingConfig(thinking_budget=0)
)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

BASE = '/content/drive/MyDrive/Research Project Synthetic Data'
os.makedirs(f'{BASE}/generated', exist_ok=True)

train = pd.read_json(f'{BASE}/data/train.json', lines=True)
train['subject_name'] = train['subject_name'].str.strip().str.lower()
train['topic_name'] = train['topic_name'].str.strip().str.lower()

with open(f'{BASE}/Taxonomy/taxonomy_filtered.json') as f:
    taxonomy = json.load(f)

subjects = list(taxonomy.keys())
print(f"{len(subjects)} subjects loaded")
print(f"train rows: {len(train)}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
16 subjects loaded
train rows: 182822


Retrying in case of error 503

In [ ]:
def generate_with_retry(prompt, config, max_retries=5):
    for attempt in range(max_retries):
        try:
            return client.models.generate_content(
                model=MODEL_ID, contents=prompt, config=config)
        except Exception as e:
            msg = str(e).lower()
            transient = any(k in msg for k in ("503", "unavailable", "overloaded", "429"))
            if not transient or attempt == max_retries - 1:
                raise
            wait = (2 ** attempt) + random.uniform(0, 1)
            time.sleep(wait)

In [ ]:
def build_generation_prompt(subject, topics, exemplars, n_topics=3):
    chosen = random.sample(topics, min(n_topics, len(topics))) if topics else []
    topic_line = f"Focus on one of these areas: {', '.join(chosen)}." if chosen else ""
    ex_text = "\n\n".join(
        f"Question: {e['question']}\nA) {e['opa']}\nB) {e['opb']}\n"
        f"C) {e['opc']}\nD) {e['opd']}\nAnswer: {'ABCD'[int(e['cop'])-1]}"
        for e in exemplars)
    return f"""You are writing exam questions for a medical entrance examination.

Here are example questions from the subject "{subject}":

{ex_text}

Write ONE new medical multiple-choice question in the subject "{subject}".
{topic_line}
The question must be factually correct, clinically accurate, and have exactly one correct answer.

Respond in EXACTLY this format and nothing else:
Question: <your question>
A) <option>
B) <option>
C) <option>
D) <option>
Answer: <single letter>
Explanation: <one or two sentences on why that answer is correct>"""


def parse_generated(text):
    try:
        q = re.search(r'Question:\s*(.+?)\n\s*A\)', text, re.S).group(1).strip()
        opts = {}
        for L in 'ABCD':
            m = re.search(rf'{L}\)\s*(.+?)(?=\n\s*[A-D]\)|\n\s*Answer:)', text, re.S)
            opts[L] = m.group(1).strip()
        ans = re.search(r'Answer:\s*([A-D])', text).group(1)
        exp = re.search(r'Explanation:\s*(.+)', text, re.S)
        item = {"question": q, "opa": opts['A'], "opb": opts['B'],
                "opc": opts['C'], "opd": opts['D'],
                "cop": 'ABCD'.index(ans) + 1,
                "exp": exp.group(1).strip() if exp else ""}
    except (AttributeError, KeyError):
        return None

        # validation — discard malformed items
    o = [item['opa'], item['opb'], item['opc'], item['opd']]
    if any(len(x) < 2 for x in o):        return None   # empty/trivial option
    if len(set(o)) < 4:                    return None   # duplicate options
    if len(item['question']) < 20:         return None   # truncated question
    if any(x.rstrip().endswith('?') for x in o): return None  # option is a question
    return item

def critic_check(item):
    prompt = f"""Review this medical exam question for correctness.

Question: {item['question']}
A) {item['opa']}
B) {item['opb']}
C) {item['opc']}
D) {item['opd']}
Stated answer: {'ABCD'[item['cop']-1]}

Is the question clinically accurate AND is the stated answer correct?
Reply with ONLY "VALID" or "INVALID"."""
    resp = generate_with_retry(prompt, CRITIC_CONFIG)
    return "VALID" in (resp.text or "").upper()

Shuffling the options

In [ ]:
def shuffle_options(item):
    """Randomise option order so the correct answer position is uniform."""
    opts = [item['opa'], item['opb'], item['opc'], item['opd']]
    correct_text = opts[item['cop'] - 1]
    random.shuffle(opts)
    item['opa'], item['opb'], item['opc'], item['opd'] = opts
    item['cop'] = opts.index(correct_text) + 1
    return item

In [ ]:
from collections import Counter

def generate_for_subject(subject, n_questions=200, n_exemplars=3, save=True):
    fname = f"{BASE}/generated/{subject.replace(' ', '_')}.json"
    if save and os.path.exists(fname):
        print(f"SKIP {subject} — file already exists")
        return None

    topics = taxonomy[subject].get('generation_topics', [])
    pool = train[train['subject_name'] == subject]
    kept, rejected, malformed, errors = [], 0, 0, 0
    start = time.time()

    while len(kept) < n_questions:
        exemplars = pool.sample(n_exemplars).to_dict('records')
        prompt = build_generation_prompt(subject, topics, exemplars)
        try:
            resp = generate_with_retry(prompt, GEN_CONFIG)
            item = parse_generated(resp.text or "")
            if item is None:
                malformed += 1
            elif critic_check(item):          # critic sees item AS GENERATED
                item = shuffle_options(item)  # shuffle AFTER the critic
                item['subject_name'] = subject
                kept.append(item)
            else:
                rejected += 1
        except Exception as e:
            errors += 1
            print(f"  error: {type(e).__name__}")

        if len(kept) and len(kept) % 25 == 0 and len(kept) != getattr(generate_for_subject, '_last', -1):
            generate_for_subject._last = len(kept)
            mins = (time.time() - start) / 60
            print(f"  {len(kept)}/{n_questions} kept  ({mins:.1f} min)")

    mins = (time.time() - start) / 60
    dist = Counter('ABCD'[q['cop'] - 1] for q in kept)
    print(f"\n{subject}: {len(kept)} kept | {rejected} rejected | "
          f"{malformed} malformed | {errors} errors | {mins:.1f} min")
    print(f"  answer distribution: {dict(sorted(dist.items()))}")

    if save:
        pd.DataFrame(kept).to_json(fname, orient='records', lines=True)
        # logging stats separately for methods section
        stats = {"subject": subject, "kept": len(kept), "rejected": rejected,
                 "malformed": malformed, "errors": errors,
                 "minutes": round(mins, 1), "answer_dist": dict(dist)}
        with open(f"{BASE}/generated/_stats.jsonl", "a") as f:
            f.write(json.dumps(stats) + "\n")
        print(f"  saved → {fname}")
    return kept

Generating 5 questions to test the pipeline

In [ ]:
test = generate_for_subject(subjects[0], n_questions=5, save=False)

for i, q in enumerate(test, 1):
    print(f"\n--- {i} ---")
    print(q['question'])
    for L, k in zip('ABCD', ['opa','opb','opc','opd']):
        mark = " ←" if 'ABCD'[q['cop']-1] == L else ""
        print(f"  {L}) {q[k]}{mark}")
    print(f"  Why: {q['exp']}")

anatomy: 5 kept | 0 rejected | 0 malformed

--- 1 ---
A patient presenting with Wallenberg syndrome (lateral medullary syndrome) is found to have an occlusion of the posterior inferior cerebellar artery (PICA). Which of the following cranial nerve nuclei is spared in this lesion?
  A) . Which of the following cranial nerve nuclei is spared in this lesion?
  B) Nucleus ambiguus
  C) Hypoglossal nucleus ←
  D) Vestibular nuclei
  Why: The hypoglossal nucleus is located medially in the medulla and is supplied by the anterior spinal artery, meaning it is spared in lateral medullary (Wallenberg) syndrome which affects the territory of the PICA.

--- 2 ---
During a mastectomy, a surgeon must be careful to preserve the nerve that runs along the lateral thoracic wall on the superficial surface of the serratus anterior muscle. Injury to this nerve would result in which of the following clinical presentations?
  A) Inability to initiate arm abduction
  B) Winging of the scapula ←
  C) Loss of se

Generating 20 questions

In [ ]:
from collections import Counter
test = generate_for_subject(subjects[0], n_questions=20, save=False)
print(Counter('ABCD'[q['cop']-1] for q in test))

anatomy: 20 kept | 0 rejected | 0 malformed
Counter({'B': 11, 'C': 5, 'A': 4})


In [ ]:
qs = [q['question'] for q in test]
print(f"{len(set(qs))} unique of {len(qs)}")

10 unique of 10


In [ ]:
print(test)

[{'question': 'During a surgical repair of a sliding hiatal hernia, the surgeon must identify the layers of the abdominal wall and their contributions to the diaphragm. Which of the following structures is continuous with the internal spermatic fascia in males?', 'opa': 'External oblique aponeurosis', 'opb': 'Internal oblique muscle', 'opc': 'Transversus abdominis muscle', 'opd': 'Transversalis fascia', 'cop': 4, 'exp': 'The transversalis fascia of the abdominal wall continues into the deep inguinal ring to form the internal spermatic fascia in males. The external oblique aponeurosis contributes to the external spermatic fascia, while the internal oblique muscle contributes to the cremasteric fascia.', 'subject_name': 'anatomy'}, {'question': 'Which of the following muscles is the only abductor of the vocal cords in the larynx?', 'opa': 'Lateral cricoarytenoid', 'opb': 'Posterior cricoarytenoid', 'opc': 'Cricothyroid', 'opd': 'Thyroarytenoid', 'cop': 2, 'exp': 'The posterior cricoaryte

In [ ]:
print(list(enumerate(subjects)))

[(0, 'anatomy'), (1, 'biochemistry'), (2, 'dental'), (3, 'ent'), (4, 'forensic medicine'), (5, 'gynaecology & obstetrics'), (6, 'medicine'), (7, 'microbiology'), (8, 'ophthalmology'), (9, 'pathology'), (10, 'pediatrics'), (11, 'pharmacology'), (12, 'physiology'), (13, 'radiology'), (14, 'social & preventive medicine'), (15, 'surgery')]


generating data

In [ ]:
# Anatomy
generate_for_subject(subjects[0], n_questions=200)

  25/200 kept  (4.1 min)
  50/200 kept  (12.8 min)
  75/200 kept  (18.3 min)
  100/200 kept  (24.0 min)
  125/200 kept  (28.3 min)
  150/200 kept  (32.6 min)
  175/200 kept  (35.4 min)
  200/200 kept  (38.8 min)

anatomy: 200 kept | 0 rejected | 2 malformed | 0 errors | 38.8 min
  answer distribution: {'A': 56, 'B': 38, 'C': 52, 'D': 54}
  saved → /content/drive/MyDrive/Research Project Synthetic Data/generated/anatomy.json


[{'question': 'Injury to the trochlear nerve (CN IV) results in paralysis of which extraocular muscle?',
  'opa': 'Superior rectus',
  'opb': 'Lateral rectus',
  'opc': 'Superior oblique',
  'opd': 'Inferior oblique',
  'cop': 3,
  'exp': 'The trochlear nerve (CN IV) exclusively innervates the superior oblique muscle, which assists in depression, abduction, and intorsion of the eyeball. Injury to this nerve leads to vertical diplopia, particularly when looking down and inward.',
  'subject_name': 'anatomy'},
 {'question': 'Which of the following layers of the scalp is also referred to as the "dangerous area of the scalp" because it facilitates the spread of infection into the cranial cavity?',
  'opa': 'Skin',
  'opb': 'Connective tissue (dense subcutaneous)',
  'opc': 'Aponeurosis (galea aponeurotica)',
  'opd': 'Loose areolar connective tissue',
  'cop': 4,
  'exp': 'The loose areolar connective tissue layer contains emissary veins that connect the scalp veins to the intracranial ven

In [ ]:
#Biochemistry
generate_for_subject(subjects[1], n_questions=200)

  25/200 kept  (5.4 min)
  50/200 kept  (11.9 min)
  75/200 kept  (24.6 min)
  100/200 kept  (36.3 min)
  125/200 kept  (47.0 min)
  150/200 kept  (58.8 min)
  175/200 kept  (65.4 min)
  200/200 kept  (75.5 min)

biochemistry: 200 kept | 0 rejected | 9 malformed | 0 errors | 75.5 min
  answer distribution: {'A': 50, 'B': 45, 'C': 54, 'D': 51}
  saved → /content/drive/MyDrive/Research Project Synthetic Data/generated/biochemistry.json


[{'question': 'Which enzyme of the urea cycle requires N-acetylglutamate (NAG) as an obligate allosteric activator?',
  'opa': 'Carbamoyl phosphate synthetase I',
  'opb': 'Ornithine transcarbamylase',
  'opc': 'Arginase',
  'opd': 'Argininosuccinate synthetase',
  'cop': 1,
  'exp': 'Carbamoyl phosphate synthetase I (CPS I) is the rate-limiting enzyme of the urea cycle and has an absolute requirement for N-acetylglutamate as an allosteric activator to undergo a conformational change that allows substrate binding.',
  'subject_name': 'biochemistry'},
 {'question': 'Which of the following apolipoproteins is responsible for activating lecithin-cholesterol acyltransferase (LCAT) to facilitate cholesterol esterification?',
  'opa': 'Apo E',
  'opb': 'Apo B-100',
  'opc': 'Apo A-I',
  'opd': 'Apo C-II',
  'cop': 3,
  'exp': 'Apolipoprotein A-I (Apo A-I) is the major structural protein in high-density lipoprotein (HDL) and acts as the obligate activator of the enzyme lecithin-cholesterol acy

In [ ]:
generate_for_subject(subjects[2], n_questions=200)

  25/200 kept  (1.3 min)
  50/200 kept  (7.2 min)
  75/200 kept  (8.7 min)
  100/200 kept  (10.9 min)
  125/200 kept  (12.3 min)
  150/200 kept  (13.7 min)
  175/200 kept  (15.8 min)
  200/200 kept  (17.8 min)

dental: 200 kept | 0 rejected | 3 malformed | 0 errors | 17.8 min
  answer distribution: {'A': 54, 'B': 56, 'C': 50, 'D': 40}
  saved → /content/drive/MyDrive/Research Project Synthetic Data/generated/dental.json


[{'question': 'Which of the following fungal infections is classically associated with "median rhomboid glossitis" of the dorsum of the tongue?',
  'opa': 'Candidiasis',
  'opb': 'Aspergillosis',
  'opc': 'Histoplasmosis',
  'opd': 'Mucormycosis',
  'cop': 1,
  'exp': 'Median rhomboid glossitis is a clinical variant of erythematous candidiasis, characterized by a persistent, asymptomatic, red, rhomboid-shaped depapillated area on the midline of the posterior dorsum of the tongue.',
  'subject_name': 'dental'},
 {'question': 'Which of the following oral manifestations is most characteristically associated with the connective tissue disorder Systemic Lupus Erythematosus (SLE)?',
  'opa': 'Discoid-like lesions with central erythema and peripheral radiating striae',
  'opb': 'Generalized diffuse gingival enlargement with bleeding',
  'opc': 'Target-like lesions of the lip vermilion',
  'opd': 'Cobblestone appearance of the buccal mucosa',
  'cop': 1,
  'exp': 'Oral lesions in Systemic Lupu

In [ ]:
generate_for_subject(subjects[3], n_questions=200)

  25/200 kept  (2.6 min)
  50/200 kept  (4.6 min)
  error: ServerError
  error: ServerError
  75/200 kept  (8.1 min)
  100/200 kept  (13.7 min)
  125/200 kept  (18.3 min)
  150/200 kept  (21.7 min)
  175/200 kept  (23.6 min)
  200/200 kept  (26.1 min)

ent: 200 kept | 0 rejected | 9 malformed | 2 errors | 26.1 min
  answer distribution: {'A': 39, 'B': 48, 'C': 63, 'D': 50}
  saved → /content/drive/MyDrive/Research Project Synthetic Data/generated/ent.json


[{'question': 'Which of the following cells in the organ of Corti are responsible for active cochlear amplification (the "cochlear amplifier")?',
  'opa': "Hensen's cells",
  'opb': 'Outer hair cells',
  'opc': "Deiters' cells",
  'opd': 'Inner hair cells',
  'cop': 2,
  'exp': 'Outer hair cells possess electromotility, which allows them to change their length in response to electrical signals, actively amplifying sound vibrations within the cochlea. Inner hair cells, on the other hand, are the primary sensory receptors that transmit auditory signals to the brain.',
  'subject_name': 'ent'},
 {'question': 'Which of the following muscles of the middle ear is supplied by the mandibular branch of the trigeminal nerve?',
  'opa': 'Levator veli palatini',
  'opb': 'Stapedius',
  'opc': 'Tensor veli palatini',
  'opd': 'Tensor tympani',
  'cop': 4,
  'exp': 'The tensor tympani muscle, which resides in the middle ear and attaches to the malleus, is embryologically derived from the first phary

In [ ]:
generate_for_subject(subjects[4], n_questions=200)

  25/200 kept  (3.7 min)
  50/200 kept  (9.1 min)
  75/200 kept  (14.0 min)
  100/200 kept  (19.4 min)
  125/200 kept  (24.4 min)
  150/200 kept  (28.4 min)
  175/200 kept  (32.8 min)
  200/200 kept  (39.0 min)

forensic medicine: 200 kept | 0 rejected | 13 malformed | 0 errors | 39.0 min
  answer distribution: {'A': 55, 'B': 54, 'C': 45, 'D': 46}
  saved → /content/drive/MyDrive/Research Project Synthetic Data/generated/forensic_medicine.json


[{'question': 'The presence of "Gettler\'s test" (estimation of chloride content in chamber blood) is classically used to support the diagnosis of drowning. In a case of freshwater drowning, what finding is expected regarding the chloride levels in the heart?',
  'opa': 'Chloride concentration is higher in the left atrium than the right atrium',
  'opb': 'Chloride concentration is lower in the left ventricle than the right ventricle',
  'opc': 'Chloride concentration is equal in both ventricles',
  'opd': 'Chloride concentration is higher in the right ventricle than the left atrium',
  'cop': 2,
  'exp': 'In freshwater drowning, inhalation of hypotonic freshwater into the lungs leads to its rapid absorption into the pulmonary circulation, diluting the blood entering the left side of the heart. Consequently, the chloride concentration in the left ventricle becomes significantly lower than that in the right ventricle.',
  'subject_name': 'forensic medicine'},
 {'question': 'A 32-year-old

In [ ]:
generate_for_subject(subjects[5], n_questions=200)

  25/200 kept  (3.5 min)
  50/200 kept  (6.9 min)
  75/200 kept  (10.1 min)
  100/200 kept  (13.9 min)
  125/200 kept  (18.1 min)
  150/200 kept  (23.0 min)
  175/200 kept  (25.4 min)
  200/200 kept  (33.9 min)

gynaecology & obstetrics: 200 kept | 0 rejected | 16 malformed | 0 errors | 33.9 min
  answer distribution: {'A': 43, 'B': 63, 'C': 44, 'D': 50}
  saved → /content/drive/MyDrive/Research Project Synthetic Data/generated/gynaecology_&_obstetrics.json


[{'question': 'A 17-year-old female presents with primary amenorrhoea. Physical examination reveals normal breast development (Tanner stage 4), but absent pubic and axillary hair. Karyotype analysis reveals a 46,XY constitution. Which of the following is the most likely diagnosis?',
  'opa': 'Mayer-Rokitansky-Küster-Hauser (MRKH) syndrome',
  'opb': 'Swyer syndrome',
  'opc': 'Turner syndrome',
  'opd': 'Complete Androgen Insensitivity Syndrome (CAIS)',
  'cop': 4,
  'exp': 'Complete Androgen Insensitivity Syndrome (CAIS) is characterized by a 46,XY karyotype, normal breast development due to peripheral conversion of testosterone to estrogen, and absent or sparse pubic and axillary hair due to end-organ resistance to androgens. In contrast, patients with MRKH syndrome have a 46,XX karyotype and normal female pubic hair distribution.',
  'subject_name': 'gynaecology & obstetrics'},
 {'question': 'Which of the following is the most appropriate initial treatment for a clinically stable pa

In [ ]:
generate_for_subject(subjects[6], n_questions=200)

  25/200 kept  (2.2 min)
  50/200 kept  (4.7 min)
  75/200 kept  (7.4 min)
  100/200 kept  (9.3 min)
  125/200 kept  (10.5 min)
  150/200 kept  (12.3 min)
  175/200 kept  (13.4 min)
  200/200 kept  (15.3 min)

medicine: 200 kept | 0 rejected | 19 malformed | 0 errors | 15.3 min
  answer distribution: {'A': 44, 'B': 57, 'C': 43, 'D': 56}
  saved → /content/drive/MyDrive/Research Project Synthetic Data/generated/medicine.json


[{'question': 'A 54-year-old male with a 15-year history of type 2 diabetes mellitus presents for a routine follow-up. His blood pressure is 138/84 mmHg and HbA1c is 7.4%. Urinalysis reveals moderately increased albuminuria (urine albumin-to-creatinine ratio of 150 mg/g). Which of the following histopathological changes is the earliest specific electron microscopic finding of diabetic nephropathy?',
  'opa': 'Subepithelial immune complex deposits',
  'opb': 'Glomerular basement membrane thickening',
  'opc': 'Nodular glomerulosclerosis (Kimmelstiel-Wilson nodules)',
  'opd': 'Diffuse mesangial sclerosis',
  'cop': 2,
  'exp': 'Glomerular basement membrane (GBM) thickening and mesangial expansion are the earliest morphologic changes of diabetic nephropathy detectable by electron microscopy, preceding the development of nodular glomerulosclerosis. Nodular glomerulosclerosis (Kimmelstiel-Wilson lesions) is highly specific but occurs later in the course of the disease.',
  'subject_name': 

In [ ]:
generate_for_subject(subjects[7], n_questions=200)

  25/200 kept  (4.4 min)
  50/200 kept  (6.5 min)
  75/200 kept  (8.9 min)
  100/200 kept  (11.0 min)
  125/200 kept  (14.3 min)
  150/200 kept  (17.2 min)
  175/200 kept  (20.4 min)
  200/200 kept  (23.0 min)

microbiology: 200 kept | 0 rejected | 8 malformed | 0 errors | 23.0 min
  answer distribution: {'A': 54, 'B': 53, 'C': 50, 'D': 43}
  saved → /content/drive/MyDrive/Research Project Synthetic Data/generated/microbiology.json


[{'question': 'A 38-year-old male with a history of kidney transplantation presents with high fever, a non-productive cough, headache, and severe watery diarrhea. On physical examination, he is bradycardic despite his elevated temperature. Laboratory findings reveal significant hyponatremia and elevated liver transaminases. Urinary antigen testing is positive. Which of the following is a characteristic feature of the causative pathogen?',
  'opa': 'It is an obligate intracellular pathogen that stains positive with acid-fast stain',
  'opb': 'It is a naked, single-stranded DNA virus that replicates in erythroid progenitor cells',
  'opc': 'It requires cysteine and iron for growth on charcoal yeast extract agar',
  'opd': 'It contains ergosterol in its cell wall and reproduces by budding',
  'cop': 3,
  'exp': 'The clinical presentation of high fever, atypical pneumonia, diarrhea, relative bradycardia, hyponatremia, and a positive urinary antigen test is highly characteristic of *Legione

In [ ]:
generate_for_subject(subjects[8], n_questions=200)

  25/200 kept  (1.1 min)
  50/200 kept  (3.2 min)
  75/200 kept  (6.0 min)
  100/200 kept  (7.7 min)
  125/200 kept  (9.0 min)
  150/200 kept  (10.4 min)
  175/200 kept  (11.4 min)
  200/200 kept  (13.4 min)

ophthalmology: 200 kept | 0 rejected | 6 malformed | 0 errors | 13.4 min
  answer distribution: {'A': 49, 'B': 48, 'C': 43, 'D': 60}
  saved → /content/drive/MyDrive/Research Project Synthetic Data/generated/ophthalmology.json


[{'question': "Fleischer's ring, a clinical feature seen in keratoconus, is caused by the deposition of which of the following in the cornea?",
  'opa': 'Calcium',
  'opb': 'Copper',
  'opc': 'Iron',
  'opd': 'Lipids',
  'cop': 3,
  'exp': "Fleischer's ring is a partial or complete ring of iron deposition (hemosiderin) located in the basal epithelium of the cornea, surrounding the base of the cone in keratoconus.",
  'subject_name': 'ophthalmology'},
 {'question': 'A 14-year-old boy presents with a tall stature, arachnodactyly, and a sudden decrease in vision in both eyes. On slit-lamp examination, bilateral subluxation of the lens (ectopia lentis) is noted. The lenses are displaced superotemporally, and the zonules are intact. Which of the following is the most likely diagnosis?',
  'opa': 'Sulfituria',
  'opb': 'Marfan syndrome',
  'opc': 'Homocystinuria',
  'opd': 'Weill-Marchesani syndrome',
  'cop': 2,
  'exp': 'Marfan syndrome is characterized by superotemporal subluxation of the

In [ ]:
generate_for_subject(subjects[9], n_questions=200)

  25/200 kept  (2.1 min)
  50/200 kept  (3.6 min)
  75/200 kept  (5.4 min)
  100/200 kept  (7.1 min)
  125/200 kept  (9.4 min)
  150/200 kept  (10.4 min)
  175/200 kept  (12.6 min)
  200/200 kept  (14.7 min)

pathology: 200 kept | 0 rejected | 12 malformed | 0 errors | 14.7 min
  answer distribution: {'A': 35, 'B': 55, 'C': 53, 'D': 57}
  saved → /content/drive/MyDrive/Research Project Synthetic Data/generated/pathology.json


[{'question': "A 55-year-old male presents to the emergency department with severe, crushing substernal chest pain radiating to his left arm. An ECG shows ST-segment elevation in leads V1 to V4. If this patient's coronary artery occlusion is not relieved, which of the following microscopic changes is most characteristically observed in the infarcted myocardium between 12 to 24 hours after the event?",
  'opa': 'Abundant granulation tissue with active neovascularization',
  'opb': 'Total removal of necrotic debris by macrophages',
  'opc': 'Infiltration by dense collagenous scar tissue',
  'opd': 'Coagulative necrosis with contraction bands and early neutrophilic infiltrate',
  'cop': 4,
  'exp': 'Between 12 to 24 hours post-myocardial infarction, the infarcted tissue characteristically exhibits ongoing coagulative necrosis, marginal contraction bands, and the initiation of a neutrophilic infiltrate. Granulation tissue, macrophage phagocytosis, and collagen scar formation occur much lat

In [ ]:
generate_for_subject(subjects[10], n_questions=200)

  25/200 kept  (1.7 min)
  50/200 kept  (3.0 min)
  75/200 kept  (5.0 min)
  100/200 kept  (6.4 min)
  125/200 kept  (9.0 min)
  150/200 kept  (10.5 min)
  175/200 kept  (11.9 min)
  200/200 kept  (13.3 min)

pediatrics: 200 kept | 0 rejected | 12 malformed | 0 errors | 13.3 min
  answer distribution: {'A': 47, 'B': 49, 'C': 52, 'D': 52}
  saved → /content/drive/MyDrive/Research Project Synthetic Data/generated/pediatrics.json


[{'question': 'A 4-year-old child presents with high fever, headache, and altered mental status. Lumbar puncture reveals cerebrospinal fluid with high protein, low glucose, and a marked neutrophilic pleocytosis. Latex agglutination test is positive for capsular polysaccharide antigen. Which of the following is the most common bacterial cause of meningitis in this age group in countries with high conjugate vaccine coverage?',
  'opa': 'Neisseria meningitidis',
  'opb': 'Haemophilus influenzae type b',
  'opc': 'Streptococcus pneumoniae',
  'opd': 'Group B Streptococcus',
  'cop': 1,
  'exp': 'Following the widespread introduction of the pneumococcal conjugate vaccine (PCV) and Hib vaccine, Neisseria meningitidis has become the most common cause of bacterial meningitis in children aged 2 to 18 years. Streptococcus pneumoniae remains a major cause but has significantly declined in vaccinated populations, while Group B Streptococcus is primarily a pathogen of the neonatal period.',
  'subj

In [ ]:
generate_for_subject(subjects[11], n_questions=200)

  25/200 kept  (1.8 min)
  50/200 kept  (6.7 min)
  75/200 kept  (8.7 min)
  100/200 kept  (11.4 min)
  125/200 kept  (13.8 min)
  150/200 kept  (15.6 min)
  175/200 kept  (17.2 min)
  200/200 kept  (18.6 min)

pharmacology: 200 kept | 0 rejected | 15 malformed | 0 errors | 18.6 min
  answer distribution: {'A': 42, 'B': 50, 'C': 48, 'D': 60}
  saved → /content/drive/MyDrive/Research Project Synthetic Data/generated/pharmacology.json


[{'question': 'Which of the following antidepressants is contraindicated in a patient being treated with linezolid due to an increased risk of serotonin syndrome?',
  'opa': 'Duloxetine',
  'opb': 'Trazodone',
  'opc': 'Mirtazapine',
  'opd': 'Bupropion',
  'cop': 1,
  'exp': 'Linezolid is a weak, reversible monoamine oxidase inhibitor (MAOI) that can cause serotonin syndrome when co-administered with serotonergic drugs. Duloxetine, a serotonin-norepinephrine reuptake inhibitor (SNRI), significantly increases serotonin levels and should not be used concomitantly with linezolid.',
  'subject_name': 'pharmacology'},
 {'question': 'Which of the following hypolipidemic drugs acts by inhibiting the Niemann-Pick C1-Like 1 (NPC1L1) transporter in the brush border of the enterocytes?',
  'opa': 'Ezetimibe',
  'opb': 'Colesevelam',
  'opc': 'Alirocumab',
  'opd': 'Gemfibrozil',
  'cop': 1,
  'exp': 'Ezetimibe selectively inhibits the NPC1L1 transporter in the small intestine, which leads to a r

In [ ]:
generate_for_subject(subjects[12], n_questions=200)

  25/200 kept  (1.6 min)
  50/200 kept  (3.4 min)
  75/200 kept  (4.8 min)
  100/200 kept  (6.4 min)
  125/200 kept  (7.6 min)
  150/200 kept  (8.4 min)
  175/200 kept  (9.9 min)
  200/200 kept  (11.1 min)

physiology: 200 kept | 0 rejected | 5 malformed | 0 errors | 11.1 min
  answer distribution: {'A': 51, 'B': 50, 'C': 45, 'D': 54}
  saved → /content/drive/MyDrive/Research Project Synthetic Data/generated/physiology.json


[{'question': 'What is the primary site of aldosterone-regulated sodium reabsorption in the renal tubule?',
  'opa': 'Thick ascending limb of the loop of Henle',
  'opb': 'Principal cells of the late distal tubule and collecting duct',
  'opc': 'Descending limb of the loop of Henle',
  'opd': 'Proximal convoluted tubule',
  'cop': 2,
  'exp': 'Aldosterone acts primarily on the principal cells of the late distal convoluted tubule and cortical collecting duct to increase the expression of apical epithelial sodium channels (ENaC) and basolateral Na+/K+ ATPase pumps, promoting sodium reabsorption.',
  'subject_name': 'physiology'},
 {'question': 'During the process of smooth muscle contraction, the binding of calcium to calmodulin directly leads to which of the following events?',
  'opa': 'Phosphorylation of the myosin light chain by myosin light chain kinase',
  'opb': 'Depolarization of the sarcolemma',
  'opc': 'Dephosphorylation of the myosin cross-bridges by myosin phosphatase',
  'o

In [ ]:
generate_for_subject(subjects[13], n_questions=200)

  25/200 kept  (1.3 min)
  50/200 kept  (3.4 min)
  75/200 kept  (5.5 min)
  100/200 kept  (6.9 min)
  125/200 kept  (8.5 min)
  150/200 kept  (9.9 min)
  175/200 kept  (11.3 min)
  200/200 kept  (12.9 min)

radiology: 200 kept | 0 rejected | 12 malformed | 0 errors | 12.9 min
  answer distribution: {'A': 53, 'B': 47, 'C': 51, 'D': 49}
  saved → /content/drive/MyDrive/Research Project Synthetic Data/generated/radiology.json


[{'question': 'Which of the following is the most sensitive ultrasound feature for diagnosing adenomyosis?',
  'opa': 'Presence of subendometrial echogenic nodules',
  'opb': 'Myometrial cysts',
  'opc': 'Thickening of the junctional zone to greater than 12 mm',
  'opd': 'Asymmetrical thickening of the myometrium',
  'cop': 2,
  'exp': 'While all options are recognized features of adenomyosis, the presence of myometrial cysts is the most specific and sensitive ultrasound sign for diagnosing this condition. Transvaginal ultrasound demonstrates high diagnostic accuracy when these tiny anechoic spaces are visualized within the myometrium.',
  'subject_name': 'radiology'},
 {'question': 'A 45-year-old male with a history of alcohol abuse presents with severe epigastric pain radiating to the back, nausea, and vomiting. A contrast-enhanced CT (CECT) scan of the abdomen is performed. The radiologist notes a well-circumscribed, fluid-density collection lacking a defined epithelial wall, locate

In [ ]:
generate_for_subject(subjects[14], n_questions=200)

  25/200 kept  (2.1 min)
  50/200 kept  (3.7 min)
  75/200 kept  (5.3 min)
  100/200 kept  (6.8 min)
  125/200 kept  (8.1 min)
  150/200 kept  (9.8 min)
  175/200 kept  (11.5 min)
  200/200 kept  (13.2 min)

social & preventive medicine: 200 kept | 0 rejected | 27 malformed | 0 errors | 13.2 min
  answer distribution: {'A': 56, 'B': 47, 'C': 44, 'D': 53}
  saved → /content/drive/MyDrive/Research Project Synthetic Data/generated/social_&_preventive_medicine.json


[{'question': 'Incineration ash of biomedical waste should be disposed of in which of the following?',
  'opa': 'Yellow bag',
  'opb': 'Red bag',
  'opc': 'Black container (Secured landfill)',
  'opd': 'Blue bag',
  'cop': 3,
  'exp': 'According to the Biomedical Waste Management Rules, incineration ash is toxic and should be disposed of in a secured landfill (historically designated in black containers/bags for municipal or inert waste disposal). Yellow bags are used to collect the waste destined for incineration, not the final ash.',
  'subject_name': 'social & preventive medicine'},
 {'question': 'In a study assessing the incubation period of tetanus in a population, the data is found to have a long tail to the right, meaning a few patients have very long incubation periods. Which of the following is true regarding this distribution?',
  'opa': 'The mean, median, and mode are all equal',
  'opb': 'The mean is greater than the median',
  'opc': 'It represents a negatively skewed dist

In [ ]:
generate_for_subject(subjects[15], n_questions=200)

  25/200 kept  (1.2 min)
  50/200 kept  (2.2 min)
  75/200 kept  (3.6 min)
  100/200 kept  (4.7 min)
  125/200 kept  (5.9 min)
  150/200 kept  (7.5 min)
  175/200 kept  (9.2 min)
  200/200 kept  (10.2 min)

surgery: 200 kept | 0 rejected | 10 malformed | 0 errors | 10.2 min
  answer distribution: {'A': 47, 'B': 46, 'C': 59, 'D': 48}
  saved → /content/drive/MyDrive/Research Project Synthetic Data/generated/surgery.json


[{'question': 'During a thyroidectomy, a surgeon accidentally ligates the superior thyroid artery too far proximally from the upper pole of the thyroid gland. Which of the following muscles is most likely to have its nerve supply compromised as a result?',
  'opa': 'Thyrohyoid',
  'opb': 'Posterior cricoarytenoid',
  'opc': 'Sternothyroid',
  'opd': 'Cricothyroid',
  'cop': 4,
  'exp': 'The external branch of the superior laryngeal nerve runs in close proximity to the superior thyroid artery and supplies the cricothyroid muscle. To avoid injuring this nerve during a thyroidectomy, the superior thyroid artery must be ligated as close to the upper pole of the thyroid gland as possible.',
  'subject_name': 'surgery'},
 {'question': 'A 62-year-old male presents with dysphagia to both solids and liquids, regurgitation of undigested food, and weight loss over the past 6 months. A barium swallow reveals a dilated esophagus with a "bird\'s beak" appearance at the lower esophageal sphincter (LE

In [ ]:
import glob
files = sorted(glob.glob(f'{BASE}/generated/*.json'))
print(f"{len(files)} subject files")

all_data = []
for f in files:
    df = pd.read_json(f, lines=True)
    print(f"  {f.split('/')[-1]:35} {len(df):4} questions")
    all_data.append(df)

pool = pd.concat(all_data, ignore_index=True)
print(f"\nTotal: {len(pool)}")
print(f"Subjects: {pool['subject_name'].nunique()}")
print(f"Answer distribution: {pool['cop'].value_counts().sort_index().to_dict()}")
print(f"Duplicate questions: {pool['question'].duplicated().sum()}")

16 subject files
  anatomy.json                         200 questions
  biochemistry.json                    200 questions
  dental.json                          200 questions
  ent.json                             200 questions
  forensic_medicine.json               200 questions
  gynaecology_&_obstetrics.json        200 questions
  medicine.json                        200 questions
  microbiology.json                    200 questions
  ophthalmology.json                   200 questions
  pathology.json                       200 questions
  pediatrics.json                      200 questions
  pharmacology.json                    200 questions
  physiology.json                      200 questions
  radiology.json                       200 questions
  social_&_preventive_medicine.json    200 questions
  surgery.json                         200 questions

Total: 3200
Subjects: 16
Answer distribution: {1: 775, 2: 806, 3: 796, 4: 823}
Duplicate questions: 51
